In [59]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langgraph.checkpoint.memory import MemorySaver

In [60]:
#Specialised reducer
from langgraph.graph.message import add_messages

In [61]:
from typing import Literal, TypedDict, Optional, Annotated

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [62]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.7, max_output_tokens=512)

In [63]:
def chat_with_ai(state: ChatState):
    messages = state['messages']
    prompt_template = PromptTemplate(
        input_variables=["messages"],
        template="You are a helpful assistant. Continue the conversation based on the following messages. Reply in short-oneline if possi:\n\n{messages}"
    )
    chain = prompt_template | llm | StrOutputParser()
    response = chain.invoke({"messages" : messages})
    return  {"messages": [AIMessage(content=response)]}
    

In [64]:
graph = StateGraph(ChatState)
checkpointer = MemorySaver()

In [65]:
graph.add_node('chat_node' , chat_with_ai)

#add_edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

In [67]:
workflow = graph.compile(checkpointer=checkpointer)

In [48]:
input_state = {
    "messages" : [
        SystemMessage(content="You are a Pre-historic animals expert. Be precise and concise in your answers."),
        HumanMessage(content="What is the largest dinosaur that ever lived?")
    ]
}

In [49]:
final_state = workflow.invoke(input_state)

In [50]:
for message in final_state['messages']:
    print(f"{message.type}: {message.content}")

system: You are a Pre-historic animals expert. Be precise and concise in your answers.
human: What is the largest dinosaur that ever lived?
ai: The largest known dinosaur is *Argentinosaurus*, weighing around 70-100 metric tons.


In [53]:
chat_history = [SystemMessage(content="You are a helpful assistant. Be precise and concise in your answers. Whenever possible, reply in short one-line answers.")]

In [68]:
thread_id = "user_1"

In [71]:
while True:
    input_text = input()
    print(f"User: {input_text}")
    list_of_words = input_text.strip().lower().split(" ")
    end_words = ["exit", "quit", "bye"]
    if any(word in list_of_words for word in end_words):
        print("Exiting the chat. Goodbye!")
        break

    config = {"configurable" : {"thread_id" : thread_id}}
    final_state = workflow.invoke({"messages": [HumanMessage(content=input_text)] }, config=config)

    print(f"AI: {final_state['messages'][-1].content}")


User: what is my name
AI: Your name is Harsh!
User: ok bye
Exiting the chat. Goodbye!


In [ ]:
workflow.get_state(config= config)

StateSnapshot(values={'messages': [HumanMessage(content='My name is Harsh', additional_kwargs={}, response_metadata={}, id='4d4abb28-747a-4378-8be3-15853fefeaeb'), AIMessage(content='Nice to meet you, Harsh! How can I help you today?', additional_kwargs={}, response_metadata={}, id='dde0e0d5-fd84-41a6-9674-8abccc198d4b', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name', additional_kwargs={}, response_metadata={}, id='8ac75500-ee27-4529-a25f-7dbf7b4d0ff6'), AIMessage(content='Your name is Harsh.', additional_kwargs={}, response_metadata={}, id='9d837112-6a46-4554-b100-d10c876e05c2', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={}, id='8269065c-5eaf-477f-a9b6-5a719fde0219'), AIMessage(content='Your name is Harsh!', additional_kwargs={}, response_metadata={}, id='cd4f4e70-ac3d-4e63-b07e-55c2adf3a29f', tool_calls=[], invalid_tool_calls=[])]}, next=(), config={'configurable': {'thread_id':